In [ ]:
# =============================================================================
# 语义分割（Semantic Segmentation）—— VOC数据集预处理
# =============================================================================
# 语义分割：对图像中的每个像素进行分类，属于同一类别的像素被赋予相同的标签
# 与目标检测不同，语义分割不区分不同实例（例如：两个人都属于"人"这一类，没有实例ID区分）
# VOC2012数据集：PASCAL Visual Object Classes Challenge 2012，包含20个前景类别+1个背景类

# %matplotlib inline 是 Jupyter 的魔法命令，使得matplotlib生成的图表直接嵌入到notebook中显示
%matplotlib inline
import os
import torch
import torchvision
from d2l import torch as d2l

# =============================================================================
# 步骤1：下载VOC2012数据集
# =============================================================================
# DATA_HUB是d2l库中用于管理数据集的注册表，存储数据集名称到(下载URL, 校验hash)的映射
# 这样可以通过名称方便地获取数据集，同时确保数据完整性
#@save
d2l.DATA_HUB['voc2012'] = (d2l.DATA_URL + 'VOCtrainval_11-May-2012.tar',
                           '4e443f8a2eca6b1dac8a6c57641b67dd40621a49')

# download_extract函数会根据名称从DATA_HUB查找并下载数据，然后解压到指定子目录
# 返回解压后数据集的本地路径，避免重复下载（利用缓存机制）
voc_dir = d2l.download_extract('voc2012', 'VOCdevkit/VOC2012')

# =============================================================================
# 步骤2：读取VOC图像和标注
# =============================================================================
# VOC数据集目录结构：
# - JPEGImages/：存放原始RGB图像（.jpg格式）
# - SegmentationClass/：存放语义分割标注图（.png格式，使用调色板索引）
# - ImageSets/Segmentation/：存放训练集/验证集划分文件（train.txt, val.txt）
#@save
def read_voc_images(voc_dir, is_train=True):
    """读取所有VOC图像并标注
    
    参数:
        voc_dir: VOC数据集根目录路径
        is_train: True读取训练集，False读取验证集
    返回:
        features: 图像列表，每个元素是torch.Tensor，shape为(C, H, W)
        labels: 标注图像列表，每个元素是torch.Tensor，shape为(C, H, W)，使用RGB格式存储类别信息
    """
    # 根据is_train选择对应的划分文件，文件每行是一个图像文件名（不含扩展名）
    txt_fname = os.path.join(voc_dir, 'ImageSets', 'Segmentation',
                             'train.txt' if is_train else 'val.txt')
    # ImageReadMode.RGB确保读取的图像通道顺序为RGB（而不是BGR或其他格式）
    mode = torchvision.io.image.ImageReadMode.RGB
    with open(txt_fname, 'r') as f:
        # split()按空白字符分割，得到所有图像文件名列表
        images = f.read().split()
    features, labels = [], []
    for i, fname in enumerate(images):
        # torchvision.io.read_image直接返回torch.Tensor，shape为(C, H, W)，像素值范围[0, 255]
        features.append(torchvision.io.read_image(os.path.join(
            voc_dir, 'JPEGImages', f'{fname}.jpg')))
        # 标注图像也是RGB格式，每个像素的RGB值对应一个类别（通过VOC_COLORMAP映射）
        labels.append(torchvision.io.read_image(os.path.join(
            voc_dir, 'SegmentationClass' ,f'{fname}.png'), mode))
    return features, labels

# 读取训练集图像和标注，features是原始图像，labels是对应的分割标注图
train_features, train_labels = read_voc_images(voc_dir, True)

# =============================================================================
# 步骤3：可视化部分训练样本
# =============================================================================
# 展示前n张图像及其标注，用于直观理解数据格式
n = 5
# 将图像和标注拼接在一起，前半部分是图像，后半部分是对应的标注
imgs = train_features[0:n] + train_labels[0:n]
# permute(1,2,0)将tensor从(C, H, W)转换为(H, W, C)，这是matplotlib显示图像所需的格式
# matplotlib期望(H, W, C)，而PyTorch默认使用(C, H, W)
imgs = [img.permute(1,2,0) for img in imgs]
# show_images显示图像网格：2行n列，第一行是原图，第二行是分割标注
d2l.show_images(imgs, 2, n);

# =============================================================================
# 步骤4：定义VOC颜色映射表和类别名称
# =============================================================================
# VOC数据集中，每个类别对应一个特定的RGB颜色值
# 例如：background=[0,0,0]黑色，aeroplane=[128,0,0]深红色，bicycle=[0,128,0]深绿色
# 这种设计使得标注图像可以直观显示（人用眼睛能看出不同区域），同时能被程序解析
#@save
VOC_COLORMAP = [[0, 0, 0], [128, 0, 0], [0, 128, 0], [128, 128, 0],
                [0, 0, 128], [128, 0, 128], [0, 128, 128], [128, 128, 128],
                [64, 0, 0], [192, 0, 0], [64, 128, 0], [192, 128, 0],
                [64, 0, 128], [192, 0, 128], [64, 128, 128], [192, 128, 128],
                [0, 64, 0], [128, 64, 0], [0, 192, 0], [128, 192, 0],
                [0, 64, 128]]

# VOC_CLASSES：每个颜色对应的类别名称，索引与VOC_COLORMAP一一对应
# 索引0是background（背景），1-20是20个前景类别（飞机、自行车、鸟等）
#@save
VOC_CLASSES = ['background', 'aeroplane', 'bicycle', 'bird', 'boat',
               'bottle', 'bus', 'car', 'cat', 'chair', 'cow',
               'diningtable', 'dog', 'horse', 'motorbike', 'person',
               'potted plant', 'sheep', 'sofa', 'train', 'tv/monitor']

# =============================================================================
# 步骤5：构建颜色到类别索引的映射
# =============================================================================
# 核心思想：将RGB颜色值编码为一个整数，作为查表索引
# 编码公式：index = (R * 256 + G) * 256 + B = R*65536 + G*256 + B
# 这样每个RGB三元组对应一个唯一的整数索引（范围0到256^3-1）
# 然后创建一个大小为256^3的查找表，存储每个颜色对应的类别索引
#@save
def voc_colormap2label():
    """构建从RGB到VOC类别索引的映射
    
    返回一个长度为256^3的一维tensor，索引是RGB编码值，值是对应的类别ID
    """
    # 256^3 = 16,777,216，覆盖所有可能的RGB颜色组合
    colormap2label = torch.zeros(256 ** 3, dtype=torch.long)
    for i, colormap in enumerate(VOC_COLORMAP):
        # 将RGB编码为单一整数索引，然后在对应位置填入类别索引i
        colormap2label[
            (colormap[0] * 256 + colormap[1]) * 256 + colormap[2]] = i
    return colormap2label

#@save
def voc_label_indices(colormap, colormap2label):
    """将VOC标签中的RGB值映射到它们的类别索引
    
    参数:
        colormap: 标注图像tensor，shape为(C, H, W)，C=3（RGB通道）
        colormap2label: 颜色到类别的映射表，由voc_colormap2label()生成
    返回:
        类别索引tensor，shape为(H, W)，每个元素是该像素的类别ID（0-20）
    """
    # permute将(C, H, W)转为(H, W, C)，然后转为numpy数组便于逐像素操作
    # astype('int32')确保乘法不会溢出
    colormap = colormap.permute(1, 2, 0).numpy().astype('int32')
    # 计算每个像素的RGB编码值，shape为(H, W)
    idx = ((colormap[:, :, 0] * 256 + colormap[:, :, 1]) * 256
           + colormap[:, :, 2])
    # 通过映射表将RGB编码转换为类别索引，输出shape为(H, W)
    return colormap2label[idx]

# =============================================================================
# 步骤6：数据增强——随机裁剪
# =============================================================================
# 语义分割的数据增强需要特别注意：对图像和标注必须做相同的几何变换！
# 如果图像裁剪了左上角，标注也必须裁剪左上角，否则标签就错位了
# 因此使用RandomCrop.get_params先确定裁剪参数，然后分别应用于图像和标注
#@save
def voc_rand_crop(feature, label, height, width):
    """随机裁剪特征和标签图像
    
    参数:
        feature: 输入图像tensor，shape为(C, H, W)
        label: 标注图像tensor，shape为(C, H, W)或(H, W)
        height, width: 裁剪后的目标尺寸
    返回:
        裁剪后的图像和标注（保持对齐）
    """
    # get_params随机生成裁剪参数：起始位置(i, j)和裁剪尺寸(h, w)
    # 返回的rect是一个元组(i, j, h, w)，i是top，j是left
    rect = torchvision.transforms.RandomCrop.get_params(
        feature, (height, width))
    # 使用相同的裁剪参数分别裁剪图像和标注，确保空间对齐
    feature = torchvision.transforms.functional.crop(feature, *rect)
    label = torchvision.transforms.functional.crop(label, *rect)
    return feature, label

# 测试随机裁剪效果：对同一张图像和标注进行n次随机裁剪并显示
imgs = []
for _ in range(n):
    imgs += voc_rand_crop(train_features[0], train_labels[0], 200, 300)

# 将tensor转换为显示格式，奇数索引是原图，偶数索引是标注
imgs = [img.permute(1, 2, 0) for img in imgs]
# [::2]取所有偶数索引（原图），[1::2]取所有奇数索引（标注），然后拼接显示
d2l.show_images(imgs[::2] + imgs[1::2], 2, n);

# =============================================================================
# 步骤7：定义VOC语义分割数据集类
# =============================================================================
# 继承torch.utils.data.Dataset，实现自定义数据加载逻辑
# 这是PyTorch的标准做法，使得可以使用DataLoader进行批量加载和多进程处理
#@save
class VOCSegDataset(torch.utils.data.Dataset):
    """一个用于加载VOC数据集的自定义数据集
    
    功能：
    1. 读取VOC图像和标注
    2. 过滤掉尺寸小于crop_size的图像
    3. 对图像进行归一化预处理（使用ImageNet均值和标准差）
    4. 随机裁剪图像和标注到固定尺寸
    5. 将标注图像的RGB颜色转换为类别索引
    """

    def __init__(self, is_train, crop_size, voc_dir):
        """
        参数:
            is_train: True表示训练集，False表示验证集
            crop_size: 裁剪尺寸元组(height, width)，模型输入的固定尺寸
            voc_dir: VOC数据集根目录路径
        """
        # 使用ImageNet预训练模型的标准化参数进行归一化
        # mean和std是ImageNet数据集的三通道均值和标准差
        # 这是迁移学习的标准做法：保持与预训练模型一致的数据预处理
        self.transform = torchvision.transforms.Normalize(
            mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        self.crop_size = crop_size
        # 读取图像和标注
        features, labels = read_voc_images(voc_dir, is_train=is_train)
        # 过滤后分别进行归一化和保存
        self.features = [self.normalize_image(feature)
                         for feature in self.filter(features)]
        self.labels = self.filter(labels)
        # 预先生成颜色到类别的映射表，避免每次__getitem__重复计算
        self.colormap2label = voc_colormap2label()
        print('read ' + str(len(self.features)) + ' examples')

    def normalize_image(self, img):
        """对图像进行归一化：先将像素值从[0,255]缩放到[0,1]，再用ImageNet参数标准化"""
        return self.transform(img.float() / 255)

    def filter(self, imgs):
        """过滤掉尺寸小于crop_size的图像
        
        原因：如果图像比crop_size小，无法进行RandomCrop，会报错
        tensor的shape是(C, H, W)，所以img.shape[1]=height, img.shape[2]=width
        """
        return [img for img in imgs if (
            img.shape[1] >= self.crop_size[0] and
            img.shape[2] >= self.crop_size[1])]

    def __getitem__(self, idx):
        """获取第idx个样本
        
        返回:
            feature: 裁剪后的图像tensor，shape为(3, crop_h, crop_w)，已归一化
            label: 裁剪后的标注tensor，shape为(crop_h, crop_w)，值为类别索引
        """
        # 随机裁剪图像和标注（保持对齐）
        feature, label = voc_rand_crop(self.features[idx], self.labels[idx],
                                       *self.crop_size)
        # 将标注图像的RGB颜色转换为类别索引矩阵
        return (feature, voc_label_indices(label, self.colormap2label))

    def __len__(self):
        """返回数据集大小"""
        return len(self.features)
    
# 创建训练集和测试集实例
crop_size = (320, 480)  # 裁剪尺寸：高320，宽480
voc_train = VOCSegDataset(True, crop_size, voc_dir)
voc_test = VOCSegDataset(False, crop_size, voc_dir)

# =============================================================================
# 步骤8：创建DataLoader进行批量加载
# =============================================================================
batch_size = 64
train_iter = torch.utils.data.DataLoader(voc_train, batch_size, shuffle=True,
                                    drop_last=True,
                                    num_workers=d2l.get_dataloader_workers())
# 测试代码：检查一个batch的数据形状
# X是图像batch，shape应为(batch_size, 3, 320, 480)
# Y是标注batch，shape应为(batch_size, 320, 480)，每个元素是类别索引
# for X, Y in train_iter:
#     print(X.shape)
#     print(Y.shape)
#     break
    
# =============================================================================
# 步骤9：封装完整的数据加载函数
# =============================================================================
# 将上述所有步骤封装为一个函数，方便复用
#@save
def load_data_voc(batch_size, crop_size):
    """加载VOC语义分割数据集
    
    这是一个便捷函数，封装了数据下载、预处理、DataLoader创建的全部流程
    
    参数:
        batch_size: 每个batch的样本数
        crop_size: 裁剪尺寸元组(height, width)
    返回:
        train_iter: 训练集DataLoader
        test_iter: 测试集DataLoader
    """
    # 下载并解压数据集（如果本地不存在）
    voc_dir = d2l.download_extract('voc2012', os.path.join(
        'VOCdevkit', 'VOC2012'))
    num_workers = d2l.get_dataloader_workers()
    # 创建训练集DataLoader，shuffle=True进行随机打乱，drop_last=True丢弃不完整的最后一个batch
    train_iter = torch.utils.data.DataLoader(
        VOCSegDataset(True, crop_size, voc_dir), batch_size,
        shuffle=True, drop_last=True, num_workers=num_workers)
    # 创建测试集DataLoader，通常不需要shuffle
    test_iter = torch.utils.data.DataLoader(
        VOCSegDataset(False, crop_size, voc_dir), batch_size,
        drop_last=True, num_workers=num_workers)
    return train_iter, test_iter